# Tiền xử lý và chia tập theo tên miền — Tuần 4

Notebook tiếp nối [01_eda.ipynb](01_eda.ipynb). Dữ liệu đầu vào là tệp PhiUSIIL gốc, với nhãn **0 = lừa đảo** và **1 = hợp lệ**. Mọi phân tích URL chạy trên chuỗi tại chỗ.

**Chạy lại:** mở thư mục dự án trong VS Code, chọn kernel Python 3.12 có `pandas`, `scikit-learn`, `tldextract` và `ipykernel`, rồi chọn **Run All**.

## 1. Kiểm tra và làm sạch

- Loại bản ghi thiếu URL, sai cú pháp HTTP(S) cơ bản hoặc có nhãn ngoài 0–1, vì các dòng đó không đủ đầu vào để phân loại có giám sát.
- Với các URL giống hệt nhau và cùng nhãn, giữ lần xuất hiện đầu tiên. Nếu một URL có cả hai nhãn, loại toàn bộ các dòng của URL đó vì chưa có căn cứ để chọn nhãn.
- Giữ URL hợp lệ có độ dài hoặc số ký tự khác thường vì chúng có thể mang tín hiệu phân loại. Giữ phân bố lớp quan sát được và báo cáo tỷ lệ từng tập. Cột dùng để phân tích là URL và nhãn, còn dữ liệu nguồn vẫn được lưu trong CSV gốc.

In [1]:
from pathlib import Path
import ipaddress
import re
import string
from urllib.parse import urlsplit

import numpy as np
import pandas as pd
from IPython.display import display
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
import tldextract

data_dir = Path("data")
csv_path = data_dir / "PhiUSIIL_Phishing_URL_Dataset.csv"
raw = pd.read_csv(csv_path, usecols=["URL", "label"], dtype={"URL": "string"},
                  keep_default_na=False, na_values=[""])
raw.index.name = "source_row"

def get_valid_host(value):
    if pd.isna(value) or any(char.isspace() or ord(char) < 32 for char in value):
        return None
    try:
        parts = urlsplit(value)
        if parts.scheme.lower() not in {"http", "https"} or not parts.hostname:
            return None
        _ = parts.port
        host = parts.hostname.rstrip(".").lower()
        try:
            return str(ipaddress.ip_address(host))
        except ValueError:
            ascii_host = host.encode("idna").decode("ascii")
            valid_labels = all(re.fullmatch(r"[a-z0-9](?:[a-z0-9-]{0,61}[a-z0-9])?", label)
                               for label in ascii_host.split("."))
            return ascii_host if valid_labels and len(ascii_host) <= 253 else None
    except (ValueError, UnicodeError):
        return None

hosts = raw["URL"].map(get_valid_host)
eligible = raw.loc[hosts.notna() & raw["label"].isin([0, 1])].copy()
conflict = eligible.groupby("URL")["label"].transform("nunique").gt(1)
consistent = eligible.loc[~conflict]
clean = consistent.drop_duplicates(subset="URL", keep="first").copy()
clean["host"] = hosts.loc[clean.index]

assert len(raw) - len(clean) == (len(raw) - len(eligible)) + conflict.sum() + (len(consistent) - len(clean))
assert clean["URL"].is_unique and clean["host"].notna().all()
assert clean["label"].isin([0, 1]).all()

## 2. Đối chiếu trước và sau xử lý

Số URL dài bất thường dùng cùng một cận trên IQR từ dữ liệu gốc để đối chiếu hai cột. Đây là thống kê mô tả, còn bước tạo đặc trưng dùng toàn bộ độ dài quan sát được trong từng tập.

In [2]:
raw_lengths = raw["URL"].str.len()
q1, q3 = raw_lengths.quantile([0.25, 0.75])
length_limit = q3 + 1.5 * (q3 - q1)

def summarize(frame, frame_hosts):
    urls = frame["URL"]
    present = urls[urls.notna() & urls.str.strip().ne("")]
    return pd.Series({
        "Số bản ghi": len(frame),
        "URL thiếu hoặc chỉ có khoảng trắng": len(urls) - len(present),
        "URL sai cú pháp (trong số URL có nội dung)": frame_hosts.loc[present.index].isna().sum(),
        "Nhãn ngoài 0–1 hoặc thiếu": (~frame["label"].isin([0, 1])).sum(),
        "Dòng lặp URL (tính từ lần thứ hai)": present.duplicated().sum(),
        "URL có cả hai nhãn": frame.groupby("URL")["label"].nunique().gt(1).sum(),
        f"URL dài hơn cận IQR ({length_limit:g} ký tự)": urls.str.len().gt(length_limit).sum(),
        "Nhãn 0 — Lừa đảo": frame["label"].eq(0).sum(),
        "Nhãn 1 — Hợp lệ": frame["label"].eq(1).sum(),
        "Tỷ lệ lừa đảo (%)": round(100 * frame["label"].eq(0).mean(), 2),
    })

comparison = pd.DataFrame({"Trước": summarize(raw, hosts),
                           "Sau": summarize(clean, clean["host"])})
display(comparison.round(2))
print(f"Đã loại {len(raw) - len(clean):,} dòng. "
      f"Trong đó {len(consistent) - len(clean):,} dòng lặp cùng URL và cùng nhãn.")

Đã loại 425 dòng. Trong đó 425 dòng lặp cùng URL và cùng nhãn.


,Trước,Sau
Số bản ghi,235795.00,235370.00
URL thiếu hoặc chỉ có khoảng trắng,0.00,0.00
URL sai cú pháp (trong số URL có nội dung),0.00,0.00
Nhãn ngoài 0–1 hoặc thiếu,0.00,0.00
Dòng lặp URL (tính từ lần thứ hai),425.00,0.00
URL có cả hai nhãn,0.00,0.00
URL dài hơn cận IQR (51.5 ký tự),21942.00,21814.00
Nhãn 0 — Lừa đảo,100945.00,100520.00
Nhãn 1 — Hợp lệ,134850.00,134850.00
Tỷ lệ lừa đảo (%),42.81,42.71


## 3. Chia train, validation và test theo nhóm tên miền

Dùng tên miền đăng ký theo danh sách hậu tố công cộng đi kèm `tldextract`, gồm cả hậu tố riêng, như khi khảo sát ở notebook trước. Với địa chỉ IP và tên máy chủ chưa nhận diện được hậu tố, dùng tên máy chủ chuẩn hóa làm nhóm.

`GroupShuffleSplit` với seed 42 hướng đến tỷ lệ **70/15/15 theo số nhóm**. Tỷ lệ theo số URL có thể khác vì mỗi nhóm có số dòng khác nhau. Validation dành cho việc lựa chọn mô hình ở giai đoạn sau, còn test dành cho đánh giá cuối. Khi chia theo nhóm, chênh lệch tỷ lệ nhãn có thể xảy ra, nên bảng dưới kiểm tra và báo cáo tỷ lệ của cả ba tập.

In [3]:
extract_domain = tldextract.TLDExtract(suffix_list_urls=(), include_psl_private_domains=True)
domain_parts = {host: extract_domain(host) for host in clean["host"].unique()}
group_by_host = {host: parts.top_domain_under_public_suffix or host
                 for host, parts in domain_parts.items()}
clean["domain_group"] = clean["host"].map(group_by_host)

seed = 42
outer = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=seed)
train_idx, holdout_idx = next(outer.split(clean, groups=clean["domain_group"]))
train = clean.iloc[train_idx]
holdout = clean.iloc[holdout_idx]

inner = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=seed)
val_idx, test_idx = next(inner.split(holdout, groups=holdout["domain_group"]))
splits = {"train": train, "validation": holdout.iloc[val_idx], "test": holdout.iloc[test_idx]}

from itertools import combinations

group_sets = {name: set(frame["domain_group"]) for name, frame in splits.items()}
overlaps = pd.Series({f"{a} / {b}": len(group_sets[a] & group_sets[b])
                      for a, b in combinations(splits, 2)}, name="Số nhóm trùng")
assert overlaps.eq(0).all()
assigned = pd.concat([frame[["label"]].assign(split=name) for name, frame in splits.items()])
assert assigned.index.is_unique and assigned.index.sort_values().equals(clean.index.sort_values())
assert all(set(frame["label"]) == {0, 1} for frame in splits.values())

split_summary = pd.DataFrame({
    name: {"Số URL": len(frame), "Tỷ lệ URL (%)": round(100 * len(frame) / len(clean), 2),
           "Số nhóm tên miền": len(group_sets[name]), "Nhãn 0": frame["label"].eq(0).sum(),
           "Nhãn 1": frame["label"].eq(1).sum(),
           "Tỷ lệ lừa đảo (%)": round(100 * frame["label"].eq(0).mean(), 2)}
    for name, frame in splits.items()
}).T
display(split_summary)
display(overlaps.to_frame())

,Số URL,Tỷ lệ URL (%),Số nhóm tên miền,Nhãn 0,Nhãn 1,Tỷ lệ lừa đảo (%)
train,164785.0,70.01,138396.0,70709.0,94076.0,42.91
validation,34356.0,14.60,29656.0,14067.0,20289.0,40.94
test,36229.0,15.39,29657.0,15744.0,20485.0,43.46


,Số nhóm trùng
train / validation,0
train / test,0
validation / test,0


## 4. Tạo đặc trưng và kiểm tra phép biến đổi

- Từ chuỗi URL, tính độ dài, số chữ số và ký tự đặc biệt, nhóm dấu phân cách, số tên miền con, cờ IP và số từ khóa xuất hiện. Từ khóa là danh sách cố định `login`, `verify`, `account`, `secure`, `update`, `password`. Giá trị IP và tên miền con là đặc trưng số. Tên miền đăng ký dùng để chia nhóm.
- Mã hóa chuỗi URL bằng TF-IDF của n-gram ký tự dài 3–5. Giữ nguyên chữ hoa/thường trong n-gram để bảo toàn thông tin về đường dẫn. Giới hạn 20.000 đặc trưng và bỏ n-gram xuất hiện trong dưới 3 URL huấn luyện để kiểm soát kích thước ma trận.
- Chỉ khớp bộ chuẩn hóa đặc trưng số và bộ từ vựng/IDF trên train. Validation và test dùng lại các phép biến đổi đã khớp. Ma trận kết hợp được lưu dạng thưa trong bộ nhớ khi chạy notebook.

In [4]:
keywords = ("login", "verify", "account", "secure", "update", "password")
punctuation = set(string.punctuation)

def host_features(host):
    try:
        ipaddress.ip_address(host)
        return 0, 1
    except ValueError:
        subdomain = domain_parts[host].subdomain
        return (len(subdomain.split(".")) if subdomain else 0), 0

host_metrics = {host: host_features(host) for host in domain_parts}
numeric_columns = ["url_length", "host_length", "digit_count", "punctuation_count",
                   "dot_count", "hyphen_count", "slash_count", "query_marker_count",
                   "subdomain_count", "is_ip", "keyword_count"]

def lexical_features(frame):
    records = []
    for url, host in zip(frame["URL"], frame["host"], strict=True):
        subdomains, is_ip = host_metrics[host]
        lower_url = url.lower()
        records.append((len(url), len(host), sum(ch.isdigit() for ch in url),
                        sum(ch in punctuation for ch in url), url.count("."), url.count("-"),
                        url.count("/"), url.count("?"), subdomains, is_ip,
                        sum(keyword in lower_url for keyword in keywords)))
    return pd.DataFrame(records, columns=numeric_columns, index=frame.index, dtype=np.float32)

lexical = {name: lexical_features(frame) for name, frame in splits.items()}
scaler = StandardScaler()
vectorizer = TfidfVectorizer(analyzer="char", ngram_range=(3, 5), lowercase=False,
                             min_df=3, max_features=20_000, dtype=np.float32)

def combine(numeric, ngrams):
    return sparse.hstack([sparse.csr_matrix(numeric), ngrams], format="csr")

X = {"train": combine(scaler.fit_transform(lexical["train"]), vectorizer.fit_transform(train["URL"]))}
for name in ("validation", "test"):
    X[name] = combine(scaler.transform(lexical[name]), vectorizer.transform(splits[name]["URL"]))
y = {name: frame["label"].to_numpy() for name, frame in splits.items()}
assert int(scaler.n_samples_seen_) == len(train)
assert all(matrix.shape == (len(y[name]), len(numeric_columns) + len(vectorizer.vocabulary_))
           and np.isfinite(matrix.data).all() for name, matrix in X.items())

display(pd.DataFrame({"Số dòng": {name: matrix.shape[0] for name, matrix in X.items()},
                      "Số đặc trưng": {name: matrix.shape[1] for name, matrix in X.items()},
                      "Số phần tử khác 0": {name: matrix.nnz for name, matrix in X.items()}}))
print(f"{len(numeric_columns)} đặc trưng từ vựng và {len(vectorizer.vocabulary_):,} n-gram ký tự.")

11 đặc trưng từ vựng và 20,000 n-gram ký tự.


,Số dòng,Số đặc trưng,Số phần tử khác 0
train,164785,20011,12329587
validation,34356,20011,2537828
test,36229,20011,2679456


## 5. Lưu kết quả chia tập

`data/splits.csv` ghi `source_row` (số thứ tự dòng dữ liệu trong CSV gốc, bắt đầu từ 0 và bỏ qua dòng tiêu đề) và `split`. Dùng tệp này với đúng CSV gốc để tái sử dụng cùng cách chia. Các dòng đã loại không có trong danh sách.

Khi kiểm định chéo ở giai đoạn huấn luyện, cần giữ nhóm tên miền trong từng lượt chia và khớp lại scaler, từ vựng, IDF trên phần huấn luyện của lượt đó.

In [5]:
manifest_path = data_dir / "splits.csv"
manifest = assigned[["split"]].sort_index()
manifest.to_csv(manifest_path)
roundtrip = pd.read_csv(manifest_path, index_col="source_row")
pd.testing.assert_frame_equal(roundtrip, manifest, check_dtype=False)
print(f"Đã lưu {len(manifest):,} dòng phân tập vào {manifest_path}.")

Đã lưu 235,370 dòng phân tập vào data\splits.csv.
